好的，同学们。请坐。

今天我们来深入探讨一个在深度学习中既简单又极其有效的技术——**丢弃法**，或者你们更常听到的英文名——**Dropout**。

这门课是深度学习的核心课程，我希望你们不仅能学会怎么用它，更要理解它*为什么*能工作。这背后有非常美妙的直觉和严谨的数学解释。

---

### 1. 什么是丢弃法？一个直观的比喻

想象一下，你们是一个投资委员会的成员。现在有两种委员会：

1.  **委员会A**：每次开会，都是完全相同的10个专家。他们共事多年，彼此非常了解，以至于有时会依赖其中一两个最强者的意见，而其他人可能就不会那么独立地深入思考。
2.  **委员会B**：每次开会，都从100个专家库中**随机抽取**20个不同的专家。因为每次的成员都不同，每个专家都必须学会独立、全面地思考，不能总是指望某几个专家。最终，这个委员会的综合决策会更强健、更全面，对个别专家的错误也不那么敏感。

**丢弃法（Dropout）** 要做的就是将你们的神经网络从“委员会A”变成“委员会B”。

在每次训练迭代中，丢弃法会以一定的概率 `p`（例如，50%）**随机地、临时地**“丢弃”（即暂时移除）网络隐藏层中的一部分神经元。这意味着它们的前向传播输出被置为零，并且在这次迭代的权重更新中，这些神经元不会有任何贡献。



*图：丢弃法在训练期间随机“关闭”神经元*

---

### 2. 动机：我们为什么要使用丢弃法？

**核心问题：过拟合（Overfitting）**

神经网络是一个拥有大量参数的强大函数 approximator。就像一个记忆力超群的学生，它很容易“死记硬背”训练数据中的所有细节（包括噪声），而不是学习泛化的、有意义的模式。这导致它在训练集上表现极好，但在没见过的新数据（测试集）上表现糟糕。

这种现象在神经元众多、连接复杂的网络中尤其严重。神经元之间可能会发展出复杂的“共适应关系”（co-adaptation）——即某些神经元依赖于其他特定的神经元才能发挥作用，而不是自己学习有用的特征。这使得模型非常脆弱。

**丢弃法如何解决过拟合？**

1.  **打破共适应（Breaking Co-adaptation）**：通过随机丢弃神经元，我们打破了神经元之间固定的、依赖性的连接。迫使每个神经元都不能过分依赖于某几个其他的神经元。它必须靠自己也能学到一些鲁棒的特征。这就像委员会B的专家，必须自己具备真才实学。
2.  **强制学习冗余特征（Ensemble Learning in Disguise）**：一个拥有 `n` 个神经元的网络，在应用丢弃法后，每次迭代都相当于在训练一个不同的、更薄的“子网络”（sub-network）。如果有 `n` 个神经元，理论上我们可以训练 `2^n` 个可能的子网络！
    在测试阶段，我们**不使用**丢弃法，而是使用整个完整的网络。但此时，这个完整网络的权重已经被平均了指数级多个子网络的经验所调整。**这相当于把一个大委员会拆分成无数个小委员会，分别训练，最后再合并他们的智慧。** 这是一种非常高效的**模型平均（Model Averaging）** 技术。

---

### 3. 技术细节：训练 vs. 测试（至关重要！）

这是理解丢弃法最关键的一点。它的行为在训练和测试阶段是**不同**的。

#### **训练阶段（Training Phase）**

1.  **随机丢弃**：对于网络中的每个神经元（通常只应用于隐藏层，输入层有时也会用，但概率更低），我们以一个预定的**丢弃概率 `p`**（例如 0.5）来决定是否将其“关闭”。`p` 也叫丢弃率（dropout rate）。`1-p` 是该神经元被保留的概率。
2.  **缩放激活值（重要！）**：在丢弃之后，为了补偿因神经元关闭而造成的预期输出值的减少，我们需要将**保留下来的神经元的输出值乘以 `1/(1-p)`**。
    *   **为什么？** 假设没有缩放，保留下的神经元的输出期望值（Expected Output）会变成原来的 `1-p` 倍。例如 `p=0.5`，期望值就减半了。这会在测试时带来问题，因为测试时我们不丢弃神经元，输出会突然变大，网络行为会不一致。通过缩放，我们在训练时保证了输出的**期望值**与测试时保持一致。

**公式化表达：**
对于一个神经元，其输出为 `a`。在训练时，我们：
1.  从一个伯努利分布中生成一个掩码（mask）向量 `m`，其中每个元素 `m_i` 以概率 `1-p` 为 1，以概率 `p` 为 0。
2.  计算丢弃后的输出： `a_dropped = a * m`
3.  **缩放**： `a_final = a_dropped / (1-p)`

#### **测试阶段（Inference Phase）**

**不使用任何随机丢弃！** 我们使用完整的网络进行前向传播。

*   **为什么？** 我们想要一个稳定、确定的预测。我们不希望每次预测的结果因为随机性而波动。
*   **那缩放呢？** 因为我们训练时已经通过除以 `(1-p)` 进行了缩放，保证了神经元输出的期望值与测试时一致，所以测试时我们什么都不用做，直接使用原始的输出 `a` 即可。网络的权重已经在训练过程中被调整到了适合“无丢弃”模式的状态。

**另一种等价实现（在某些框架中常见）**：在测试时，将所有权重乘以 `(1-p)`。这样训练时就不需要缩放激活值了。但这是一种效率较低的做法，因为它要求测试时改变模型。现在主流做法是**训练时缩放，测试时不动**。

---

### 4. 数学视角：一种自适应正则化

从正则化的角度看，丢弃法可以被视为一种在均方误差损失函数上增加的**自适应正则项**。

传统的 L2 正则化是对权重的平方和进行惩罚，防止权重变得过大。而丢弃法的正则化效果更强：它不仅仅是对权重进行惩罚，而是在每次迭代中**粗暴地阻止某些神经元参与学习**。这相当于对一个巨大的子网络集合进行正则化。

其效果是让网络的权重普遍地变得更小、更分散，而不是让少数权重变得特别大。这提升了模型的泛化能力。

---

### 5. 如何在代码中实现？

在现代深度学习框架中，实现丢弃法非常简单。以下是概念性代码：

**PyTorch:**
```python
import torch
import torch.nn as nn

class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
        self.layer1 = nn.Linear(784, 256)
        self.layer2 = nn.Linear(256, 128)
        self.layer3 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(p=0.5)  # 定义一个丢弃层，p=0.5

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.dropout(x)  # 在激活函数后应用丢弃
        x = torch.relu(self.layer2(x))
        x = self.dropout(x)  # 再次应用
        x = self.layer3(x)
        return x

# 训练循环
model = MyNet()
model.train()  # 训练模式 -> Dropout 生效

# ... 训练代码 ...

# 测试循环
model.eval()  # 评估模式 -> Dropout 被关闭
with torch.no_grad():
    # ... 测试代码 ...
```

**TensorFlow/Keras:**
```python
from tensorflow.keras import models, layers

model = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dropout(0.5),  # 直接作为一个层添加
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10)
])

# Keras 在 model.fit() 时自动启用训练行为（Dropout生效），
# 在 model.evaluate() 或 model.predict() 时自动关闭。
```

---

### 6. 超参数与最佳实践

*   **丢弃概率 `p`**：这是最主要的超参数。
    *   **输入层**：通常使用较低的丢弃率（如 0.1 或 0.2），或者不用。
    *   **隐藏层**：常用值是 **0.5**。这是一个很好的起点。
    *   **输出层**：通常**不使用**丢弃法。
*   **网络宽度**：应用丢弃法后，你可以并且应该使用**更宽**的网络（每层更多的神经元）。因为每次只使用一部分神经元，更大的宽度保证了有足够的容量来学习。
*   **与其它正则化器结合**：丢弃法通常与其它技术如 **L2 权重衰减** 和 **批标准化（BatchNorm）** 一起使用。请注意，Dropout 和 BatchNorm 一起使用时有时会相互影响，需要小心调参，但通常它们能很好地协同工作。
*   **趋势**：在非常深的网络（如ResNet）中，由于BatchNorm的存在和架构本身的正则化能力，Dropout的使用比以前少了。但在全连接层较多的网络中（如Transformer、MLP），它仍然是不可或缺的核心技术。

### 总结

同学们，丢弃法是一个优雅而强大的工具。它的核心思想是：

> **通过在训练中引入随机性来防止复杂系统对特定路径的过度依赖，从而强制学习到更鲁棒、更泛化的特征。它在训练时平均多个子网络，在测试时整合为一个大而强的网络。**

希望这个解释能帮助你们不仅理解其操作，更领悟其思想。下次课我们将探讨另一种正则化技术——批标准化。下课！

有任何问题，可以在Office Hour来找我。

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device {device}")

batch_size = 128
lr = 0.001
epochs = 10
dropout_prob = 0.5

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
    ])

train_dataset = torchvision.datasets.MNIST(root = './data',
                                           train=True,
                                           transform=transform,
                                           download=True)

train_loader = DataLoader(dataset=train_dataset,batch_size=batch_size,shuffle=True)

test_dataset = torchvision.datasets.MNIST(root = './data',
                                           train=False,
                                           transform=transform,
                                           download=True)

test_loader = DataLoader(dataset=test_dataset,batch_size=batch_size,shuffle=False)


class NeuralNetWithDroput(nn.Module) : 
    def __init__(self,input_size,hidden_size1,hidden_size2,num_classes,dropout_prob):
        super(NeuralNetWithDroput,self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_prob)  # 定义Dropout层
        
    def forward(self,x) :
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        
        out = self.fc2(out)
        out = self.relu(out)
        out = self.dropout(out)
        
        out = self.fc3(out)
        return out
    
input_size = 28 * 28
output_size = 10
hidden_size1 = 256
hidden_size2 = 128

model = NeuralNetWithDroput(input_size,hidden_size1,hidden_size2,output_size,dropout_prob).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = lr)


def train_model(model,train_loader,critrtion,opt,epochs) :
    model.train()
    total_step = len(train_loader)
    for epoch in range(epochs):
        start_time = time.time()
        total_loss = 0
        correct = 0
        total = 0
        
        for i,(images,labels) in enumerate(train_loader):
            images = images.reshape(-1, 28*28).to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs,labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data,1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if (i+1) % 100 == 0 :
                print(f'Epoch [{epoch+1}/{epochs}], Step [{i+1}/{total_step}], Loss: {loss.item():.4f}')

            epoch_time = time.time() - start_time
            accuracy = 100 * correct / total
            avg_loss = total_loss / total_step
            print(f'Epoch [{epoch+1}/{epochs}] completed in {epoch_time:.2f}s, '
              f'Average Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')
            print('-' * 60)
    
    
# 测试函数
def test_model(model, test_loader):
    model.eval()  # 设置为评估模式（禁用Dropout）
    with torch.no_grad():  # 禁用梯度计算
        correct = 0
        total = 0
        for images, labels in test_loader:
            images = images.reshape(-1, 28*28).to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        accuracy = 100 * correct / total
        print(f'Test Accuracy of the model on the 10000 test images: {accuracy:.2f}%')
    return accuracy

# 训练模型
print("Starting training...")
train_model(model, train_loader, criterion, optimizer, epochs)
    
print("\nTesting model...")
test_accuracy = test_model(model, test_loader)

# 保存模型
torch.save(model.state_dict(), 'model_with_dropout.pth')
print("Model saved as 'model_with_dropout.pth'")


class NeuralNetWithoutDropout(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, num_classes):
        super(NeuralNetWithoutDropout, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, num_classes)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.relu(self.fc2(out))
        out = self.fc3(out)
        return out

# 训练并测试没有Dropout的模型
model_no_dropout = NeuralNetWithoutDropout(input_size, hidden_size1, hidden_size2, output_size).to(device)
optimizer_no_dropout = optim.Adam(model_no_dropout.parameters(), lr=lr)

print("Training model without dropout...")
train_model(model_no_dropout, train_loader, criterion, optimizer_no_dropout, epochs)

print("Testing model without dropout...")
test_accuracy_no_dropout = test_model(model_no_dropout, test_loader)

print(f"\nComparison Results:")
print(f"With Dropout:    {test_accuracy:.2f}%")
print(f"Without Dropout: {test_accuracy_no_dropout:.2f}%")
print(f"Improvement:     {test_accuracy - test_accuracy_no_dropout:.2f}%")

using device cuda
Starting training...
Epoch [1/10] completed in 0.51s, Average Loss: 0.0050, Accuracy: 7.03%
------------------------------------------------------------
Epoch [1/10] completed in 0.53s, Average Loss: 0.0098, Accuracy: 11.33%
------------------------------------------------------------
Epoch [1/10] completed in 0.54s, Average Loss: 0.0146, Accuracy: 14.58%
------------------------------------------------------------
Epoch [1/10] completed in 0.56s, Average Loss: 0.0193, Accuracy: 16.21%
------------------------------------------------------------
Epoch [1/10] completed in 0.58s, Average Loss: 0.0238, Accuracy: 17.97%
------------------------------------------------------------
Epoch [1/10] completed in 0.59s, Average Loss: 0.0282, Accuracy: 19.79%
------------------------------------------------------------
Epoch [1/10] completed in 0.61s, Average Loss: 0.0324, Accuracy: 22.43%
------------------------------------------------------------
Epoch [1/10] completed in 0.63s